###### robot_objects

In [1]:
# FuelTank class

from dataclasses import dataclass, field
from math import inf


@dataclass(frozen=True)
class FuelTank:
    level: float = field(default=-inf)
    cost: float = field(default=0.25)
    volume: float = field(default=100)

    def __post_init__(self):
        if self.level > self.volume:
            raise ValueError("Fuel tank level cannot be greater than volume")
        if self.level < 0:
            object.__setattr__(self, "level", self.volume)
        if self.cost <= 0 or self.volume <= 0:
            raise ValueError("Cost and volume must be positive.")

    def drop_fuel(self, distance: float):
        new_level = self.level - (distance * self.cost)
        if new_level < 0:
            raise ValueError("Fuel tank level cannot be negative")
        return FuelTank(level=new_level, cost=self.cost, volume=self.volume)

    def refuel(self):
        # takes time can stop mid-way
        ...
        if self.is_full():
            raise ValueError("Fuel tank is full")

        return FuelTank(level=self.volume, cost=self.cost, volume=self.volume)

    def is_empty(self):
        return self.level == 0

    def is_full(self):
        return self.level == self.volume
        

In [2]:
# Degrees class

from dataclasses import dataclass, field
from math import radians, isclose, isinf, isnan
from functools import lru_cache

class InvalidAngleError(Exception):
    """Custom exception for invalid angle values."""
    pass

@dataclass(frozen=True)
class Degrees:
    _angle: float = field(repr=False)

    def __post_init__(self):
        # Normalize the angle using the setter


        if not isinstance(self._angle, (int, float)) or isinf(self.angle) or isnan(self.angle):
            raise InvalidAngleError

        object.__setattr__(self, '_angle', self._normalize_angle(self._angle))

    @property
    def angle(self) -> float:
        """Get the normalized angle."""
        return self._angle

    @staticmethod
    @lru_cache(maxsize=None)
    def _normalize_angle(angle: float) -> float:
        """Normalize the angle to be within [0, 360) degrees."""
        return angle % 360

    def turn_left(self, degrees: float = 90) -> 'Degrees':
        """Turn left by the given number of degrees."""
        return Degrees(self._normalize_angle(self.angle - degrees))

    def turn_right(self, degrees: float = 90) -> 'Degrees':
        """Turn right by the given number of degrees."""
        return Degrees(self._normalize_angle(self.angle + degrees))

    def to_radians(self) -> float:
        """Convert degrees to radians."""
        return radians(self.angle)

    def __add__(self, other: 'Degrees') -> 'Degrees':
        """Add two Degrees instances."""
        if not isinstance(other, Degrees): #
            raise ValueError(f"Invalid addition: {self.angle} + {other.angle} results in an invalid angle.")
        return Degrees(self._normalize_angle(self.angle + other.angle))

    def __sub__(self, other: 'Degrees') -> 'Degrees':
        """Subtract two Degrees instances."""
        if not isinstance(other, Degrees): #
            raise ValueError(f"Invalid subtraction: {self.angle} - {other.angle} results in an invalid angle.")

        return Degrees(self._normalize_angle(self.angle - other.angle))

    def __eq__(self, other: object) -> bool:
        """Check equality of two Degrees instances."""
        if not isinstance(other, Degrees):
            return NotImplemented
        return isclose(self.angle, other.angle, abs_tol=1e-9)

    def __hash__(self) -> int:
        """Return a hash of the angle for hashable collections."""
        return hash(round(self.angle, 9))

    def __repr__(self) -> str:
        return f"Degrees(angle={self.angle})"

    def __str__(self) -> str:
        return f"{self.angle}°"

    def to_cardinal(self) -> str:
        """Convert angle to cardinal direction."""
        cardinal_points = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
        index = round(self.angle / 45) % 8
        return cardinal_points[index]
        

In [3]:
# Position class

from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from math import sqrt, isclose, cos, sin, isnan, isinf

# from src.main.world_objects.robot_objects.degrees import Degrees


class InvalidPositionError(Exception):
    """Custom exception for invalid position coordinates."""
    pass


@dataclass(frozen=True)
class Position:
    x: float
    y: float

    def __post_init__(self):
        # Check if coordinates are numeric
        if not isinstance(self.x, (int, float)) or not isinstance(self.y, (int, float)) or isnan(self.x) or isnan(self.y) or isinf(self.x) or isinf(self.y):  #
            raise InvalidPositionError(
                f"Coordinates must be numeric, got x: {type(self.x).__name__}, y: {type(self.y).__name__}."
            )

    @lru_cache(maxsize=None)
    def distance_to(self, other: "Position") -> float:
        """Calculate the distance to another Position."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)

    def move(self, angle: "Degrees"| int, steps: float) -> "Position":
        """Move the position by a certain number of steps in the specified angle."""
        if not isinstance(angle, Degrees):  #
            raise ValueError("Angle must be a Degrees instance.")

        rad_angle = angle.to_radians()
        return Position(self.x + steps * cos(rad_angle), self.y + steps * sin(rad_angle))

    def is_in(self, top_left: "Position", bottom_right: "Position") -> bool:
        """Check if the position is within a defined rectangular area."""
        if not all(isinstance(p, Position) for p in [top_left, bottom_right]):  #
            raise ValueError("Top left and bottom right must be Position instances.")
        return (
            top_left.x <= self.x <= bottom_right.x
            and bottom_right.y <= self.y <= top_left.y
        )

    def __add__(self, other: "Position") -> "Position":
        """Add two positions."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return Position(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Position") -> "Position":
        """Subtract two positions."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return Position(self.x - other.x, self.y - other.y)

    def __eq__(self, other: object) -> bool:
        """Check equality of two positions."""
        if not isinstance(other, Position):
            return NotImplemented
        return isclose(self.x, other.x, abs_tol=1e-9) and isclose(
            self.y, other.y, abs_tol=1e-9
        )

    def surrounding(self) -> ["Position"]:
        return [
            (self.x - 1, self.y + 1), (self.x + 1, self.y - 1),  # alt diag
            (self.x, self.y + 1), (self.x, self.y - 1),  # vertical
            (self.x + 1, self.y), (self.x - 1, self.y), # horizontal
            (self.x + 1, self.y + 1), (self.x - 1, self.y - 1),  # main diag
        ]

    def as_tuple(self):
        return self.x, self.y

    def __repr__(self):
        return f"Position(x={self.x}, y={self.y})"

    def __str__(self) -> str:
        return f"({self._format_value(self.x)}, {self._format_value(self.y)})"

    @staticmethod
    def _format_value(value: float) -> str:
        """Format value for string representation."""
        if abs(value - round(value)) < 1e-6:
            return f"{round(value)}"
        else:
            return f"{value:.6f}"
            

In [4]:
# Shield class

from dataclasses import dataclass, field
import threading
from typing import Optional

@dataclass(frozen=True)
class Shield:
    level: Optional[float] = field(default=None)  # Input shield level
    shield_max: int = field(default=5)  # Default max shield level
    repair_delay: int = field(default=5)   # Default repair delay
    _is_repairing: threading.Event = field(default_factory=threading.Event, init=False)
    lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        # Normalize level to be >= 0 and <= shield_max
        if self.level is None:
            object.__setattr__(self, 'level', self.shield_max)
        else:
            normalized_level = min(max(self.level, 0), self.shield_max)
            object.__setattr__(self, 'level', normalized_level)

        object.__setattr__(self, '_is_repairing', False)  # Initialize repairing status

    @property
    def is_repairing(self) -> bool:
        """Check if the shield is currently being repaired."""
        with self.lock:
            return self._is_repairing

    def repair_shield(self) -> None:
        """Initiate a repair of the shield after a delay."""
        with self.lock:
            current_level = self.level if self.level is not None else 0
            if current_level < self.shield_max and not self._is_repairing:
                object.__setattr__(self, '_is_repairing', True)
                threading.Timer(self.repair_delay, self._finish_repair).start()

    def _finish_repair(self) -> None:
        """Finish repairing the shield."""
        with self.lock:
            object.__setattr__(self, 'level', self.shield_max)
            object.__setattr__(self, '_is_repairing', False)

    def damage_shield(self, damage: float) -> 'Shield':
        """Apply damage to the shield, returning a new Shield instance with updated level."""
        with self.lock:
            current_level = self.level if self.level is not None else 0
            new_level = max(current_level - damage, 0)
            return Shield(level=new_level, shield_max=self.shield_max, repair_delay=self.repair_delay)

    def __eq__(self, other: object) -> bool:
        """Check if two shields are equal based on their levels and max values."""
        if not isinstance(other, Shield):
            return NotImplemented
        return (self.level == other.level and 
                self.shield_max == other.shield_max)

    def __hash__(self) -> int:
        """Return the hash based on shield level and max."""
        return hash((self.level, self.shield_max))

    def __str__(self) -> str:
        return (f"Shield Level: {self.level}/{self.shield_max}, "
                f"Repairing: {self.is_repairing}")

    def __lt__(self, other: 'object') -> bool:
        """Compare shields based on their shield levels."""
        if not isinstance(other, Shield):
            return NotImplemented
        s = self.level if self.level is not None else 0
        o = other.level if other.level is not None else 0
        return s < o
        

In [5]:
# Weapon class

import threading
from dataclasses import dataclass, field

# Custom exception class
class WeaponError(Exception):
    """Base class for weapon-related exceptions."""
    def __init__(self, message: str):
        super().__init__(message)

@dataclass(frozen=True)
class Weapon:
    _ammo_max: int = field(default=5)
    _ammo: int = field(default=_ammo_max)
    _damage: float = field(default=1.0)
    _loading: bool = field(default=False)
    _load_delay: float = field(default=5.0)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        if self._ammo > self._ammo_max:
            raise ValueError("Ammo cannot exceed ammo_max")

    @property
    def ammo(self) -> int:
        return self._ammo

    # @ammo.setter
    # def ammo(self, value):
    #     # return Weapon(_ammo=value, _ammo_max=self._ammo_max, _damage=self._damage, _loading=self._loading, _load_delay=self._load_delay)
    #     self._ammo = value

    @property
    def ammo_max(self) -> int:
        return self._ammo_max

    @property
    def damage(self) -> float:
        return self._damage

    @property
    def loading(self) -> bool:
        return self._loading

    @property
    def load_delay(self) -> float:
        return self._load_delay

    @property
    def is_empty(self) -> bool:
        return self.ammo == 0

    @property
    def can_reload(self) -> bool:
        return not self._loading and self._ammo < self._ammo_max

    def shot(self) -> 'Weapon':
        if self.is_empty:
            raise WeaponError("Cannot shoot: Out of ammo, current ammo: {}".format(self.ammo))
        
        with self._lock:
            return Weapon(_ammo=self._ammo - 1, _ammo_max=self._ammo_max, _damage=self._damage,
                          _loading=self._loading, _load_delay=self._load_delay)

    def reload(self) -> 'Weapon':
        if self.loading:
            raise WeaponError("Already loading. Wait for the current reload to finish.")
        
        with self._lock:
            return Weapon(_ammo=self._ammo_max, _ammo_max=self._ammo_max, _damage=self._damage,
                          _loading=True, _load_delay=self._load_delay)

    def __hash__(self):
        return hash((self._ammo, self._ammo_max, self._damage, self._loading, self._load_delay))


##### robot

In [6]:
# Robot class

from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

# from src.main.world_objects.robot_objects.fuel_tank import FuelTank
# from src.main.world_objects.robot_objects.position import Position
# from src.main.world_objects.robot_objects.degrees import Degrees
# from src.main.world_objects.robot_objects.shield import Shield
# from src.main.world_objects.robot_objects.weapon import Weapon, WeaponError


class RobotType(Enum):
    SCOUT = "scout"
    SNIPER = "sniper"
    TANK = "tank"
    ASSAULT = "assault"
    SUPPORT = "support"


@dataclass
class Robot:
    name: str = field(default="bot")
    position: Position = field(default_factory=lambda: Position(0, 0))
    direction: Degrees = field(default_factory=lambda: Degrees(0))
    shield: Shield = field(default_factory=lambda: Shield(shield_max=5))
    weapon: Weapon = field(default_factory=lambda: Weapon(_ammo=5))
    tank: FuelTank = field(default_factory=lambda: FuelTank(volume=50))
    robot_type: RobotType = field(default=RobotType.SUPPORT)

    def __post_init__(self):
        self.set_attributes_based_on_type()

    def set_attributes_based_on_type(self):
        # Define attributes based on the RobotType Enum
        type_attributes = {
            RobotType.SCOUT: (1, 1, 1, 1, 1),
            RobotType.SNIPER: (2, 2, 2, 2, 2),
            RobotType.TANK: (3, 3, 3, 3, 3),
            RobotType.ASSAULT: (4, 4, 4, 4, 4),
            RobotType.SUPPORT: (5, 5, 5, 5, 5),
        }

        # Normalize robot_type first
        if isinstance(self.robot_type, str):
            try:
                self.robot_type = RobotType(self.robot_type.lower())
            except ValueError:
                allowed = [t.value for t in RobotType]
                print(f"Invalid robot type: '{self.robot_type}'. Must be one of: {allowed}")
                self.robot_type = RobotType.SUPPORT

        # Look up attributes based on the robot's type
        attributes = type_attributes.get(self.robot_type)
        if attributes:
            shot_damage, ammo_max, shield_max, repair_delay, reload_delay = attributes
            self.weapon = Weapon(
                _ammo=ammo_max, _load_delay=reload_delay, _damage=shot_damage, _ammo_max=ammo_max)
            self.shield = Shield(shield_max=shield_max,
                                 repair_delay=repair_delay)

    def update_position(self, nr_steps: int, forward: bool) -> bool:
        steps = nr_steps if forward else -nr_steps
        
        if self._drop_fuel(nr_steps):
            self.position = self.position.move(self.direction, steps)
            return True
        return False

    def move_forward(self, nr_steps: int) -> bool:
        return self.update_position(nr_steps, forward=True)

    def move_backward(self, nr_steps: int) -> bool:
        return self.update_position(nr_steps, forward=False)

    def turn_left(self, degrees: float = 90) -> None:
        self.direction = self.direction.turn_left(degrees)

    def turn_right(self, degrees: float = 90) -> None:
        self.direction = self.direction.turn_right(degrees)

    def damage_shield(self, damage: float) -> None:
        self.shield = self.shield.damage_shield(damage)

    def repair_shield(self) -> None:
        self.shield.repair_shield()

    def shoot(self) -> None:
        try:
            self.weapon = self.weapon.shot()
        except WeaponError as e:
            print(f"{e}")

    def reload(self) -> None:
        try:
            self.weapon = self.weapon.reload()
        except WeaponError as e:
            print(e)

    def refuel(self) -> None:
        try:
            self.tank = self.tank.refuel()
        except ValueError as e:
            print(e)

    def shield_level(self) -> Optional[int]:
        return self.shield.level

    def tank_level(self) -> float:
        return self.tank.level

    def _drop_fuel(self, steps: int):
        try:
            self.tank = self.tank.drop_fuel(distance=steps)
            return True
        except ValueError:
            print("not enough fuel")
            return False

    def __str__(self):
        return (
            f"Name: {self.name}, Position: {self.position}, Direction: {self.direction.angle}, "
            f"Shield Level: {self.shield_level()}, Ammo: {self.weapon.ammo}, Fuel Level: {self.tank_level()}, Type: {self.robot_type.value}"
        )


#### world

In [7]:
# World class

from dataclasses import dataclass, field
from typing import Dict, List

# from src.main.world_objects.robot_objects.fuel_tank import FuelTank
# from src.main.world_objects.robot_objects.position import Position
# from src.main.world_objects.robot_objects.degrees import Degrees
# from src.main.world_objects.robot import Robot
# from src.main.world_objects.robot_objects.weapon import Weapon


@dataclass
class World:
    robots: Dict[str, Robot] = field(default_factory=dict)

    def spawn_robot(
        self, name: str, position: Position, direction: Degrees, robot_type: str = "basic", tank: 'FuelTank'= FuelTank()
    ) -> Robot:
        """Spawn a new robot with the given name, position, direction, and type."""
        name = name.lower()
        if name in self.robots:
            raise ValueError(f"A robot with the name '{name}' already exists.")
        self.robots[name] = Robot(
            name=name, position=position, direction=direction,robot_type=robot_type, tank=tank
        )
        return self.robots[name]

    def get_robot(self, name: str) -> Robot:
        """Get a robot by name."""
        name = name.lower()
        return self.robots.get(name)  #

    def _move_robot(self, name: str, steps: int, forward: bool) -> bool:
        """Move a robot by a certain number of steps."""
        robot = self.get_robot(name)
        print(robot.position, end=" --> ")
        if robot:
            update = robot.update_position(steps, forward)
            print(robot.position)
            return update
        return False

    def move_robot_forward(self, name: str, steps: int = 0):
        return self._move_robot(name=name, steps=steps, forward=True)
    def move_robot_back(self, name: str, steps: int = 0):
        return self._move_robot(name=name, steps=steps, forward=False)

    def turn_robot_left(self, name: str, degrees: float = 90) -> None:
        """Turn a robot left by a certain number of degrees."""
        robot = self.get_robot(name)
        if robot:
            robot.turn_left(degrees)
    def turn_robot_right(self, name: str, degrees: float = 90) -> None:
        """Turn a robot right by a certain number of degrees."""
        robot = self.get_robot(name)
        if robot:
            robot.turn_right(degrees)

    def list_robots(self) -> List[str]:
        """List all robot names in the world."""
        return list(self.robots)

    def shoot(self, shooter_name: str) -> None:
        shooter = self.get_robot(shooter_name)
        if not shooter or shooter.weapon.ammo <= 0:
            print(f"{shooter_name} cannot shoot: Out of ammo.")
            return

        # Update ammo and set status
        gun = shooter.weapon
        shooter.weapon = Weapon(gun.ammo_max,gun.ammo-1, gun.damage, gun.loading, gun.load_delay)
        # shooter.set_status(f"{shooter.name} fired weapon.")

        shooter_x, shooter_y = shooter.position.x, shooter.position.y
        shot_direction = shooter.direction
        # targets_hit = []

        for bot in self.robots.values():
            if bot.name == shooter_name:
                continue

            bot_x, bot_y = bot.position.x, bot.position.y

            # Check if bot is in the line of fire
            is_in_line_of_fire = (
                    (bot_y == shooter_y and
                     ((bot_x < shooter_x and shot_direction == Degrees(180)) or
                      (bot_x > shooter_x and shot_direction == Degrees(0)))) or
                    (bot_x == shooter_x and
                     ((bot_y < shooter_y and shot_direction == Degrees(270)) or
                      (bot_y > shooter_y and shot_direction == Degrees(90))))
            )

            if is_in_line_of_fire:
                bot.damage_shield(gun.damage)  # Apply damage, adjust as needed
                # targets_hit.append(bot.name)
                # bot.set_status(f"{bot.name} was shot.")
                return


    def __str__(self):
        return f"World with robots: {', '.join(self.list_robots())}"


In [8]:
world = World()
world

World(robots={})

In [9]:
world.spawn_robot("bot")  # should provide random defualts (position&degrees)

TypeError: World.spawn_robot() missing 2 required positional arguments: 'position' and 'direction'

In [10]:
world.spawn_robot("bot", Position(0,0)) # how to expect required positionall argumnt error anyway

TypeError: World.spawn_robot() missing 1 required positional argument: 'direction'

In [11]:
world.spawn_robot("bot", Position(0,0), Degrees(90))

Invalid robot type: 'basic'. Must be one of: ['scout', 'sniper', 'tank', 'assault', 'support']


Robot(name='bot', position=Position(x=0, y=0), direction=Degrees(angle=90), shield=Shield(level=5, shield_max=5, repair_delay=5, _is_repairing=False, lock=<unlocked _thread.lock object at 0x7275029390>), weapon=Weapon(_ammo_max=5, _ammo=5, _damage=5, _loading=False, _load_delay=5, _lock=<unlocked _thread.lock object at 0x727502a6d0>), tank=FuelTank(level=100, cost=0.25, volume=100), robot_type=<RobotType.SUPPORT: 'support'>)

In [12]:
world

World(robots={'bot': Robot(name='bot', position=Position(x=0, y=0), direction=Degrees(angle=90), shield=Shield(level=5, shield_max=5, repair_delay=5, _is_repairing=False, lock=<unlocked _thread.lock object at 0x7275029390>), weapon=Weapon(_ammo_max=5, _ammo=5, _damage=5, _loading=False, _load_delay=5, _lock=<unlocked _thread.lock object at 0x727502a6d0>), tank=FuelTank(level=100, cost=0.25, volume=100), robot_type=<RobotType.SUPPORT: 'support'>)})

In [13]:
world.move_robot_forward("bot")

(0, 0) --> (0, 0)


True

In [14]:
world.move_robot_forward("bot", 1)

(0, 0) --> (0, 1)


True

In [15]:
world.move_robot_back("bot", 1)

(0, 1) --> (0, 0)


True

In [16]:
world.spawn_robot("bot", Position(0,0), Degrees(90)) # should append a value after the name not fail

ValueError: A robot with the name 'bot' already exists.

In [ ]:
[_.position for _ in world.robots]

In [17]:
[_.position for _ in world.list_robots()]

AttributeError: 'str' object has no attribute 'position'

In [19]:
world.list_robots()

['bot', 'bot1']

In [18]:
world.spawn_robot("bot1", Position(0,0), Degrees(90))

Invalid robot type: 'basic'. Must be one of: ['scout', 'sniper', 'tank', 'assault', 'support']


Robot(name='bot1', position=Position(x=0, y=0), direction=Degrees(angle=90), shield=Shield(level=5, shield_max=5, repair_delay=5, _is_repairing=False, lock=<unlocked _thread.lock object at 0x7275062750>), weapon=Weapon(_ammo_max=5, _ammo=5, _damage=5, _loading=False, _load_delay=5, _lock=<unlocked _thread.lock object at 0x72726ff110>), tank=FuelTank(level=100, cost=0.25, volume=100), robot_type=<RobotType.SUPPORT: 'support'>)